In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
ev_charge_df = pd.read_csv("/content/ev_charging_patterns.csv")

In [ ]:
ev_charge_df.head()

,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),Time of Day,Day of Week,State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years),Charger Type,User Type
0,User_1,BMW i3,108.463007,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712346,0.591363,36.389181,13.087717,Evening,Tuesday,29.371576,86.119962,293.602111,27.947953,2.0,DC Fast Charger,Commuter
1,User_2,Hyundai Kona,100.000000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.339275,3.133652,30.677735,21.128448,Morning,Monday,10.115778,84.664344,112.112804,14.311026,3.0,Level 1,Casual Driver
2,User_3,Chevy Bolt,75.000000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.128876,2.452653,27.513593,35.667270,Morning,Thursday,6.854604,69.917615,71.799253,21.002002,2.0,Level 2,Commuter
3,User_4,Hyundai Kona,50.000000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.457824,1.266431,32.882870,13.036239,Evening,Saturday,83.120003,99.624328,199.577785,38.316313,1.0,Level 1,Long-Distance Traveler
4,User_5,Hyundai Kona,50.000000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.629104,2.019765,10.215712,10.161471,Morning,Saturday,54.258950,63.743786,203.661847,-7.834199,1.0,Level 1,Long-Distance Traveler


In [ ]:
ev_charge_df.shape

(1320, 20)

In [ ]:
ev_charge_df.isnull().sum()

,0
User ID,0
Vehicle Model,0
Battery Capacity (kWh),0
Charging Station ID,0
Charging Station Location,0
Charging Start Time,0
Charging End Time,0
Energy Consumed (kWh),66
Charging Duration (hours),0
Charging Rate (kW),66


In [ ]:
null_cols = ['Energy Consumed (kWh)', 'Charging Rate (kW)', 'Distance Driven (since last charge) (km)']
for col in null_cols:
    ev_charge_df[col] = ev_charge_df[col].fillna(ev_charge_df[col].median())

In [ ]:
ev_charge_df.isnull().sum()

,0
User ID,0
Vehicle Model,0
Battery Capacity (kWh),0
Charging Station ID,0
Charging Station Location,0
Charging Start Time,0
Charging End Time,0
Energy Consumed (kWh),0
Charging Duration (hours),0
Charging Rate (kW),0


In [ ]:
ev_charge_df['Charging Start Time'] = pd.to_datetime(ev_charge_df['Charging Start Time'])
ev_charge_df['Charging End Time'] = pd.to_datetime(ev_charge_df['Charging End Time'])

In [ ]:
ev_charge_df['Hour'] = ev_charge_df['Charging Start Time'].dt.hour
ev_charge_df['DayOfWeek'] = ev_charge_df['Charging Start Time'].dt.dayofweek
ev_charge_df['Month'] = ev_charge_df['Charging Start Time'].dt.month
ev_charge_df['IsWeekend'] = ev_charge_df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

In [ ]:
demand_df = ev_charge_df.groupby(['Charging Station Location', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend']).agg({
    'Energy Consumed (kWh)': 'sum',
    'Temperature (°C)': 'mean',
    'Battery Capacity (kWh)': 'mean'
}).reset_index()

In [ ]:
demand_df.rename(columns={'Energy Consumed (kWh)': 'Total_Demand'}, inplace=True)

In [ ]:
le = LabelEncoder()
demand_df['Location_Encoded'] = le.fit_transform(demand_df['Charging Station Location'])

In [ ]:
hourly_trend = demand_df.groupby('Hour')['Total_Demand'].mean().reset_index()
fig1 = px.line(hourly_trend, x='Hour', y='Total_Demand',
              title='Average Hourly Charging Demand Trends',
              template='plotly_dark', line_shape='spline')
fig1.show()

In [ ]:
day_map = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}
day_trend = demand_df.groupby('DayOfWeek')['Total_Demand'].mean().reset_index()
day_trend['Day'] = day_trend['DayOfWeek'].map(day_map)
fig2 = px.bar(day_trend, x='Day', y='Total_Demand', color='Total_Demand',
             title='Demand Distribution by Day of Week', template='plotly_dark')
fig2.show()

In [ ]:
X = demand_df[['Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'Temperature (°C)', 'Battery Capacity (kWh)', 'Location_Encoded']]
y = demand_df['Total_Demand']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name} Metrics ---")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R-Squared: {r2:.2f}\n")

evaluate(y_test, rf_preds, "Random Forest")

Random Forest Metrics ---
MAE: 25.43
RMSE: 32.16
R-Squared: 0.20



In [ ]:
results = pd.DataFrame({'Actual': y_test, 'Predicted': rf_preds}).head(50).reset_index()
fig3 = px.line(results, x=results.index, y=['Actual', 'Predicted'],
              title='Demand Prediction: Actual vs Predicted (RF Model)',
              template='plotly_dark')
fig3.show()

In [ ]:
import matplotlib.pyplot as plt

location_demand = demand_df.groupby('Charging Station Location')['Total_Demand'].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.bar(location_demand['Charging Station Location'],
        location_demand['Total_Demand'],
        color='skyblue',
        edgecolor='navy')
plt.title('Total Energy Consumption by Location', fontsize=16, fontweight='bold')
plt.xlabel('Charging Station Location', fontsize=12)
plt.ylabel('Total Energy Consumed (kWh)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('location_wise_demand.png')
plt.show()